# Differential Binding & Peak Annotation

**Tier 3 — Applied Bioinformatics | Module 24 · Notebook 2**

*Prerequisites: Notebook 1 (ChIP-seq Pipeline) — deduplicated BAMs and called peaks required.*

---

**By the end of this notebook you will be able to:**
1. Run differential binding analysis with DiffBind across conditions
2. Annotate peaks to nearest genomic features with ChIPseeker
3. Visualize peak distribution across genomic regions (promoter, intron, intergenic)
4. Run GO and pathway enrichment on genes associated with differential peaks
5. Overlap ChIP-seq peaks with RNA-seq DEGs to identify regulatory targets

**Key resources:**
- [Bioconductor DiffBind vignette](https://bioconductor.org/packages/release/bioc/vignettes/DiffBind/inst/doc/DiffBind.pdf)
- [Bioconductor ChIPseeker vignette](https://bioconductor.org/packages/release/bioc/vignettes/ChIPseeker/inst/doc/ChIPseeker.html)


---[< Previous: ChIP-seq Processing Pipeline](01_chipseq_pipeline.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: ONT Data Processing >](../25_Long_Read_Sequencing/01_ont_processing.ipynb)---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import matplotlib_venn as venn_mod
    HAS_VENN = True
except ImportError:
    HAS_VENN = False

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
print('Setup complete.')


## 1. Differential Binding with DiffBind (R)

**DiffBind** quantifies reads in a consensus peak set across all samples and applies DESeq2 or edgeR to identify peaks that gain or lose binding between experimental conditions. The **consensus peak set** is constructed as the union of peaks called in at least 2 samples (default `minMembers = 2`), ensuring that only reproducible peaks are included in the analysis. This reduces noise from spurious peaks unique to a single replicate. The resulting matrix — samples × consensus peaks with read counts — is then processed identically to an RNA-seq count matrix.

The **DiffBind sample sheet** is a CSV file with one row per sample and columns specifying: `SampleID`, `Condition` (e.g., Control/Treatment), `Replicate` (integer), `bamReads` (path to deduplicated BAM), `ControlID` (matching Input sample ID), `bamControl` (path to Input BAM), `Peaks` (path to narrowPeak/broadPeak file), and `PeakCaller` (e.g., `macs`). The `dba()` function reads this sheet; `dba.count()` tallies reads in the consensus peak set; `dba.normalize()` applies TMM (edgeR) or RLE (DESeq2) normalization to correct for library size differences across samples.

`dba.contrast()` defines the pairwise comparison, referencing the `DBA_CONDITION` metadata column. `dba.analyze()` runs DESeq2 by default (set `method = DBA_EDGER` to switch). `dba.report()` returns a GRanges object with log₂ fold change, p-value, and FDR for each consensus peak; the `th` and `fold` arguments filter to a significance threshold. Volcano plots from `dba.plotVolcano()` and PCA plots from `dba.plotPCA()` quickly reveal whether replicates cluster by condition and whether differential binding is driven by a small number of peaks or genome-wide.


In [ ]:
# ============================================================
# DiffBind Differential Binding Analysis (R)
# Run in R or via rpy2 — requires R + Bioconductor packages
# ============================================================

# --- Sample sheet (samplesheet.csv) format ---
# SampleID,   Condition,  Replicate, bamReads,                   bamControl,
#   Peaks,                              PeakCaller
# CTCF_Ctrl1, Control,    1,  dedup/ctcf_ctrl1_dedup.bam, dedup/input1.bam,
#   peaks/ctcf_ctrl1_peaks.narrowPeak,  macs
# CTCF_Ctrl2, Control,    2,  dedup/ctcf_ctrl2_dedup.bam, dedup/input2.bam,
#   peaks/ctcf_ctrl2_peaks.narrowPeak,  macs
# CTCF_Trt1,  Treatment,  1,  dedup/ctcf_trt1_dedup.bam,  dedup/input3.bam,
#   peaks/ctcf_trt1_peaks.narrowPeak,   macs
# CTCF_Trt2,  Treatment,  2,  dedup/ctcf_trt2_dedup.bam,  dedup/input4.bam,
#   peaks/ctcf_trt2_peaks.narrowPeak,   macs

# library(DiffBind)
#
# # 1. Load sample sheet and build DBA object
# dba_obj <- dba(sampleSheet = 'samplesheet.csv')
# print(dba_obj)
#
# # 2. Count reads in consensus peak set (peaks in >= 2 samples)
# dba_obj <- dba.count(dba_obj, bUseSummarizeOverlaps = TRUE)
#
# # 3. Normalize using RLE (same as DESeq2 default)
# dba_obj <- dba.normalize(dba_obj, normalize = DBA_NORM_RLE)
#
# # 4. Define contrast: Treatment vs. Control
# dba_obj <- dba.contrast(dba_obj, categories = DBA_CONDITION, minMembers = 2)
#
# # 5. Run differential analysis (DESeq2 by default)
# dba_obj <- dba.analyze(dba_obj, method = DBA_DESEQ2)
#
# # 6. Report results (FDR < 0.05, |FC| > 1.5)
# db_peaks <- dba.report(dba_obj, th = 0.05, fold = log2(1.5))
# print(db_peaks)
#
# # 7. Volcano plot
# dba.plotVolcano(dba_obj)
#
# # 8. PCA of binding affinity scores
# dba.plotPCA(dba_obj, DBA_CONDITION, label = DBA_ID)
#
# # Export to BED
# rtracklayer::export(db_peaks, 'diffbind_results.bed')
print('R code shown as comments — run in an R environment or Rscript.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simulate DiffBind output (as would come from dba.report())
np.random.seed(42)
n_peaks = 45000  # consensus peak set size

log2fc = np.concatenate([
    np.random.normal(0, 0.5, 42000),   # unchanged
    np.random.normal(2.5, 0.6, 1500),  # gained binding (treatment)
    np.random.normal(-2.2, 0.6, 1500), # lost binding (treatment)
])

# p-values: most non-significant, some very significant
pval_base = np.random.exponential(0.3, n_peaks)
pval_base[:3000] = np.random.uniform(1e-30, 0.001, 3000)  # true positives
pval = np.clip(pval_base, 1e-50, 1)
fdr = np.minimum(pval * n_peaks / (np.argsort(np.argsort(pval)) + 1), 1)

df_diffbind = pd.DataFrame({
    'log2FC': log2fc[:n_peaks],
    'pvalue': pval,
    'FDR': fdr,
    'Peak': [f'peak_{i}' for i in range(n_peaks)]
})

df_diffbind['neg_log10_FDR'] = -np.log10(df_diffbind['FDR'].clip(1e-50))
df_diffbind['Significant'] = (
    (df_diffbind['FDR'] < 0.05) & (df_diffbind['log2FC'].abs() > np.log2(1.5))
)
df_diffbind['Direction'] = 'Unchanged'
df_diffbind.loc[
    df_diffbind['Significant'] & (df_diffbind['log2FC'] > 0), 'Direction'
] = 'Gained'
df_diffbind.loc[
    df_diffbind['Significant'] & (df_diffbind['log2FC'] < 0), 'Direction'
] = 'Lost'

n_gained = (df_diffbind['Direction'] == 'Gained').sum()
n_lost   = (df_diffbind['Direction'] == 'Lost').sum()
print(f'Consensus peaks: {n_peaks:,}')
print(f'Gained binding sites: {n_gained:,} ({100*n_gained/n_peaks:.1f}%)')
print(f'Lost binding sites:   {n_lost:,} ({100*n_lost/n_peaks:.1f}%)')
print(f'Unchanged:            {(df_diffbind["Direction"] == "Unchanged").sum():,}')

color_map = {'Gained': 'coral', 'Lost': 'steelblue', 'Unchanged': 'lightgray'}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for direction, grp in df_diffbind.groupby('Direction'):
    axes[0].scatter(
        grp['log2FC'], grp['neg_log10_FDR'],
        c=color_map[direction],
        s=3 if direction == 'Unchanged' else 6,
        alpha=0.5 if direction == 'Unchanged' else 0.8,
        label=f'{direction} (n={len(grp):,})'
    )

axes[0].axhline(-np.log10(0.05), color='black', linestyle='--',
                lw=1, alpha=0.7, label='FDR = 0.05')
axes[0].axvline(np.log2(1.5), color='gray', linestyle=':', lw=1)
axes[0].axvline(-np.log2(1.5), color='gray', linestyle=':', lw=1)
axes[0].set_xlabel('log2 Fold Change (Treatment / Control)')
axes[0].set_ylabel('-log10(FDR)')
axes[0].set_title('DiffBind Volcano Plot\nDifferential ChIP-seq Binding')
axes[0].legend(markerscale=2)
axes[0].set_xlim(-8, 8)

# MA plot
sig = df_diffbind['Significant']
mean_signal = np.random.exponential(5, n_peaks) + 1
axes[1].scatter(np.log2(mean_signal[~sig]), df_diffbind['log2FC'][~sig],
                c='lightgray', s=3, alpha=0.5, label='Unchanged')
axes[1].scatter(
    np.log2(mean_signal[sig & (df_diffbind['Direction'] == 'Gained')]),
    df_diffbind['log2FC'][sig & (df_diffbind['Direction'] == 'Gained')],
    c='coral', s=6, alpha=0.8, label='Gained'
)
axes[1].scatter(
    np.log2(mean_signal[sig & (df_diffbind['Direction'] == 'Lost')]),
    df_diffbind['log2FC'][sig & (df_diffbind['Direction'] == 'Lost')],
    c='steelblue', s=6, alpha=0.8, label='Lost'
)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xlabel('log2 Mean Signal')
axes[1].set_ylabel('log2 Fold Change')
axes[1].set_title('MA Plot\n(Mean vs. Fold Change)')
axes[1].legend(markerscale=2)

plt.tight_layout()
plt.show()


## 2. Peak Annotation with ChIPseeker

**ChIPseeker** annotates each ChIP-seq peak to the nearest genomic feature using a **TxDb** (transcript database) annotation package for the relevant genome assembly (e.g., `TxDb.Hsapiens.UCSC.hg38.knownGene`). It assigns each peak a feature category based on its position relative to annotated transcripts: **Promoter** (within the defined window upstream/downstream of TSS), 5′ UTR, 3′ UTR, Exon, Intron, Downstream, or Distal Intergenic. A peak is assigned to the nearest TSS by default, but multiple overlapping transcripts at the same locus may result in multiple annotations.

The `annotatePeak()` function accepts a GRanges object or a BED/narrowPeak file path. The `tssRegion = c(-2000, 200)` argument defines the promoter window as 2 kb upstream to 200 bp downstream of the TSS (reflecting that TATA-box and initiator elements are typically within 200 bp downstream). Setting `annoDb = 'org.Hs.eg.db'` automatically adds gene symbols, Entrez IDs, and Ensembl IDs to the annotation table, which are required for downstream enrichment analyses with clusterProfiler.

Visualisation in ChIPseeker conveys biology directly: `plotAnnoPie()` and `plotAnnoBar()` show the proportion of peaks in each genomic category. TF ChIP-seq peaks for activating factors (CTCF, p53) are typically 25–50% promoter-associated. Enhancer marks such as H3K27ac skew strongly toward **intronic and intergenic** regions (> 60% non-promoter), reflecting their enrichment at distal enhancers. `plotDistToTSS()` plots the distribution of distances from each peak to its nearest TSS, distinguishing proximal (< 1 kb) from distal regulatory elements.


In [ ]:
# ============================================================
# ChIPseeker Peak Annotation (R)
# ============================================================

# library(ChIPseeker)
# library(TxDb.Hsapiens.UCSC.hg38.knownGene)
# library(org.Hs.eg.db)
# library(clusterProfiler)
#
# txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
#
# # --- Annotate narrowPeak file ---
# peaks <- readPeakFile('peaks/tf_sample_peaks.narrowPeak', as = 'GRanges')
#
# anno <- annotatePeak(
#     peaks,
#     tssRegion     = c(-2000, 200),
#     TxDb          = txdb,
#     annoDb        = 'org.Hs.eg.db'
# )
#
# # Summary table
# print(anno)
#
# # Pie chart of genomic feature distribution
# plotAnnoPie(anno)
#
# # Bar chart version
# plotAnnoBar(anno)
#
# # Distance to nearest TSS histogram
# plotDistToTSS(anno,
#     title = 'Distribution of peaks relative to TSS',
#     ylab  = 'Peaks (%)'
# )
#
# # Extract annotated DataFrame
# anno_df <- as.data.frame(anno)
# head(anno_df[, c('seqnames','start','end','annotation','SYMBOL','distanceToTSS')])
#
# # Promoter peaks only (|TSS distance| < 2 kb)
# promoter_peaks <- anno_df[abs(anno_df$distanceToTSS) < 2000, ]
# cat('Promoter-associated peaks:', nrow(promoter_peaks), '\n')
print('R code shown as comments — run in an R environment or Rscript.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n_peaks = 12500  # TF ChIP-seq peaks

# Simulate genomic annotation distribution (TF-like: enriched at promoters)
annotation_categories = {
    'Promoter (<=1kb)':   0.28,
    'Promoter (1-2kb)':   0.09,
    'Promoter (2-3kb)':   0.04,
    "5' UTR":             0.02,
    "3' UTR":             0.03,
    '1st Exon':           0.02,
    'Other Exon':         0.03,
    '1st Intron':         0.10,
    'Other Intron':       0.18,
    'Downstream (<=300)': 0.01,
    'Distal Intergenic':  0.20,
}

categories = list(annotation_categories.keys())
proportions = np.array(list(annotation_categories.values()))
counts = (proportions * n_peaks).astype(int)

anno_df = pd.DataFrame({
    'Feature': categories, 'Count': counts, 'Proportion': proportions
})

print('Peak Annotation Summary (TF ChIP-seq)')
print('=' * 50)
print(anno_df.to_string(index=False))
print(f'\nTotal peaks: {counts.sum():,}')
print(f'Promoter peaks (<=3kb): {counts[:3].sum():,} '
      f'({100*counts[:3].sum()/counts.sum():.1f}%)')
print(f'Intergenic peaks: {counts[-1]:,} ({100*counts[-1]/counts.sum():.1f}%)')

# Simulate distance-to-TSS distribution
dist_promoter   = np.random.normal(0, 400, counts[:3].sum())
dist_intergenic = np.concatenate([
    np.random.uniform(-50000, -3000, counts[-1]//2),
    np.random.uniform(3000, 50000, counts[-1]//2)
])
dist_intronic   = np.random.uniform(-100000, 100000, counts[7]+counts[8])
dist_all = np.concatenate([dist_promoter, dist_intronic, dist_intergenic])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_pie = plt.cm.Set3(np.linspace(0, 1, len(categories)))
wedge_props = {'edgecolor': 'white', 'linewidth': 1.5}
axes[0].pie(counts, labels=None, colors=colors_pie,
            autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
            wedgeprops=wedge_props, startangle=90)
axes[0].set_title('Genomic Feature Distribution\n(ChIPseeker annotatePeak)')
axes[0].legend(categories, loc='lower left', bbox_to_anchor=(-0.3, -0.1),
               fontsize=7, ncol=2)

axes[1].hist(np.clip(dist_all, -50000, 50000), bins=80,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='TSS')
axes[1].axvline(-2000, color='orange', lw=1, linestyle=':', label='+-2 kb promoter')
axes[1].axvline(2000, color='orange', lw=1, linestyle=':')
axes[1].set_xlabel('Distance to TSS (bp)')
axes[1].set_ylabel('Number of peaks')
axes[1].set_title('Distance to Nearest TSS\n(plotDistToTSS)')
axes[1].legend()
axes[1].set_xlim(-50000, 50000)

plt.tight_layout()
plt.show()


## 3. GO Enrichment of Peak-Associated Genes

**Gene Ontology (GO) enrichment** analysis tests whether the genes near annotated or differential peaks are statistically overrepresented in specific biological processes (BP), molecular functions (MF), or cellular components (CC). The `clusterProfiler::enrichGO()` function applies a one-sided hypergeometric test and corrects for multiple testing using Benjamini–Hochberg (BH) FDR. Inputs are Entrez gene IDs extracted from the ChIPseeker annotation table (the `geneId` column); outputs are a ranked list of enriched GO terms with gene count, gene ratio, p-value, and adjusted p-value.

**Background gene set selection** is critical for avoiding ascertainment bias. The `universe` argument in `enrichGO()` defines the background: it should be **all genes expressed in your system** (from RNA-seq, those with CPM > 1 in at least one sample) rather than all annotated human genes. Using all 20,000 human genes as background inflates significance for housekeeping processes that are present in most gene sets. If no expression data is available, all genes with at least one annotated transcript in the TxDb used for peak annotation is a reasonable alternative.

A **dotplot** visualises the enriched GO terms: the x-axis shows the gene ratio (fraction of query genes annotated to that term), dot size encodes the number of query genes, and colour encodes the adjusted p-value. Focus on specific child terms rather than broad parent terms — 'biological process' or 'cellular process' will always appear enriched because they are ancestors of almost every specific term. `clusterProfiler::simplify()` uses semantic similarity to remove redundant terms from the result set, retaining the most specific representative term from each cluster.


In [ ]:
# ============================================================
# GO Enrichment with clusterProfiler (R)
# ============================================================

# library(clusterProfiler)
# library(org.Hs.eg.db)
# library(enrichplot)
# library(ggplot2)
#
# # Extract gene IDs from ChIPseeker annotation
# anno_df <- as.data.frame(anno)
# gene_ids <- unique(na.omit(anno_df$geneId))  # Entrez IDs
#
# # Background: all expressed genes (from RNA-seq) or all annotated genes
# background_genes <- keys(org.Hs.eg.db, keytype = 'ENTREZID')
#
# # --- GO Biological Process enrichment ---
# ego_bp <- enrichGO(
#     gene          = gene_ids,
#     universe      = background_genes,
#     OrgDb         = org.Hs.eg.db,
#     ont           = 'BP',
#     pAdjustMethod = 'BH',
#     pvalueCutoff  = 0.05,
#     qvalueCutoff  = 0.2,
#     readable      = TRUE   # convert Entrez IDs to gene symbols
# )
#
# # Simplify redundant GO terms (uses semantic similarity)
# ego_bp_simplified <- clusterProfiler::simplify(ego_bp, cutoff = 0.7, by = 'p.adjust')
#
# # Dot plot
# dotplot(ego_bp_simplified, showCategory = 20, font.size = 10) +
#     ggtitle('GO Biological Process Enrichment')
#
# # Enrichment map (network of similar terms)
# ego_bp_pairwise <- pairwise_termsim(ego_bp_simplified)
# emapplot(ego_bp_pairwise, showCategory = 30)
#
# # Gene-concept network
# cnetplot(ego_bp_simplified, showCategory = 5, foldChange = NULL)
print('R code shown as comments — run in an R environment or Rscript.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

np.random.seed(42)

# Simulate GO enrichment results (as would be returned by enrichGO)
go_terms = [
    ('GO:0006355', 'regulation of transcription, DNA-templated',       180, 0.42, 1.2e-15),
    ('GO:0045944', 'positive regulation of RNA pol II transcription',   95, 0.38, 3.4e-12),
    ('GO:0008283', 'cell population proliferation',                      88, 0.35, 8.1e-11),
    ('GO:0051726', 'regulation of cell cycle',                           72, 0.33, 2.2e-10),
    ('GO:0048523', 'negative regulation of cellular process',           120, 0.31, 5.6e-10),
    ('GO:0007049', 'cell cycle',                                          81, 0.30, 1.1e-9),
    ('GO:0006915', 'apoptotic process',                                   65, 0.29, 3.3e-9),
    ('GO:0030154', 'cell differentiation',                               110, 0.28, 7.8e-9),
    ('GO:0001501', 'skeletal system development',                         42, 0.25, 2.1e-8),
    ('GO:0007165', 'signal transduction',                                142, 0.24, 4.5e-8),
    ('GO:0045087', 'innate immune response',                              55, 0.22, 9.9e-8),
    ('GO:0006954', 'inflammatory response',                               48, 0.21, 2.3e-7),
    ('GO:0042981', 'regulation of apoptotic process',                     61, 0.20, 5.1e-7),
    ('GO:0071356', 'cellular response to tumor necrosis factor',          33, 0.19, 1.1e-6),
    ('GO:0008285', 'negative regulation of cell population prolif',       44, 0.18, 2.4e-6),
]

df_go = pd.DataFrame(
    go_terms, columns=['GO_ID', 'Description', 'Count', 'GeneRatio', 'p_adjust']
)
df_go['-log10_padj'] = -np.log10(df_go['p_adjust'])
df_go = df_go.sort_values('GeneRatio', ascending=True)

print('Top GO Biological Process Terms')
print('=' * 60)
print(df_go[['Description', 'Count', 'GeneRatio', 'p_adjust']].to_string(
    index=False, float_format='{:.2e}'.format))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Dot plot (left)
scatter = axes[0].scatter(
    df_go['GeneRatio'], range(len(df_go)),
    c=df_go['-log10_padj'], s=df_go['Count'] * 1.5,
    cmap='RdYlBu_r', vmin=5, vmax=16, edgecolors='gray', lw=0.5
)
axes[0].set_yticks(range(len(df_go)))
axes[0].set_yticklabels([t[:45] for t in df_go['Description']], fontsize=8)
axes[0].set_xlabel('Gene Ratio')
axes[0].set_title('GO Biological Process Enrichment\n(clusterProfiler enrichGO)')
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('-log10(p.adjust)')
for size in [20, 60, 120]:
    axes[0].scatter([], [], c='gray', s=size*1.5,
                    label=f'{size} genes', edgecolors='gray')
axes[0].legend(title='Gene Count', loc='lower right', fontsize=8)

# Bar chart of -log10(padj)
df_go_sorted = df_go.sort_values('-log10_padj', ascending=True).tail(10)
colors_bar = cm.RdYlBu_r(df_go_sorted['-log10_padj'] / 16)
axes[1].barh(range(len(df_go_sorted)), df_go_sorted['-log10_padj'],
             color=colors_bar, edgecolor='white')
axes[1].set_yticks(range(len(df_go_sorted)))
axes[1].set_yticklabels([t[:45] for t in df_go_sorted['Description']], fontsize=8)
axes[1].axvline(-np.log10(0.05), color='red', linestyle='--', lw=1, label='p.adj = 0.05')
axes[1].set_xlabel('-log10(adjusted p-value)')
axes[1].set_title('Top 10 Enriched GO BP Terms')
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. ChIP-seq + RNA-seq Integration

Integrating ChIP-seq peaks with RNA-seq differentially expressed genes (DEGs) reveals the **regulatory logic** underlying transcriptional changes. The central question is: are genes whose expression changes between conditions also bound by the transcription factor under study at or near their promoters? This bridges the gap between chromatin occupancy (ChIP-seq) and transcriptional output (RNA-seq), distinguishing **direct** regulatory targets (bound + differentially expressed) from **indirect** targets (changed expression without evidence of direct TF binding at their promoters).

To define regulatory targets, create **promoter BED files** for each DEG (e.g., ±2 kb around the annotated TSS) and intersect with differential ChIP-seq peaks using `bedtools intersect -wa -wb`. A gene is called a regulatory target if it has a DEG promoter overlapping a gained or lost binding site from DiffBind. To account for strand direction, use the `strand` column of the TSS BED to compute the correct upstream/downstream window. For distal regulation through enhancers, extend the window to ±50 kb or use ENCODE enhancer–promoter links (Activity-by-Contact model, ENCODE SCREEN).

To assess whether the overlap is greater than expected by chance, apply a **Fisher's exact test** using a 2×2 contingency table: (DEG ∩ diff peak | DEG only | diff peak only | neither) relative to all expressed gene promoters. A significant odds ratio confirms that the TF preferentially binds the promoters of genes that change expression. **UpSet plots** are the preferred visualisation for multi-way overlaps (e.g., DEGs, gained peaks, lost peaks, accessible chromatin from ATAC-seq), since they scale to more than 3 sets where Venn diagrams become unreadable.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- Create promoter BED file for DEGs (+/-2 kb around TSS) ---
# Input: DEG list with columns: gene_id, chrom, tss, strand

!awk 'NR>1 {
    if ($4 == "+") print $1"\t"($3-2000)"\t"($3+2000)"\t"$2"\t.\t"$4;
    else            print $1"\t"($3-2000)"\t"($3+2000)"\t"$2"\t.\t"$4
}' deg_tss.bed | bedtools sort -i - > deg_promoters_2kb.bed

# --- Intersect differential ChIP peaks with DEG promoters ---
!bedtools intersect \
    -a diffbind_results.bed \
    -b deg_promoters_2kb.bed \
    -wa -wb \
    > chip_deg_overlap.bed

!echo 'Peaks overlapping DEG promoters:'; wc -l chip_deg_overlap.bed
!echo 'Total differential peaks:'; wc -l diffbind_results.bed

# --- Also test random overlap for significance (permutation) ---
!bedtools shuffle \
    -i diffbind_results.bed \
    -g hg38.chrom.sizes \
    -noOverlapping > diffbind_shuffled.bed

!bedtools intersect \
    -a diffbind_shuffled.bed \
    -b deg_promoters_2kb.bed \
    -wa > chip_deg_overlap_random.bed

!echo 'Random overlap:'; wc -l chip_deg_overlap_random.bed


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Simulate integration results
n_degs = 3000
n_diff_peaks = 2800
n_all_genes = 20000
n_all_peaks = 45000

n_overlap_gained = 245  # gained peaks overlapping upregulated DEG promoters
n_overlap_lost   = 108  # lost peaks overlapping downregulated DEG promoters
n_total_overlap  = n_overlap_gained + n_overlap_lost

# 2x2 contingency table for Fisher's exact test
contingency = np.array([
    [n_total_overlap, n_degs - n_total_overlap],
    [n_diff_peaks - n_total_overlap,
     n_all_genes - n_degs - (n_diff_peaks - n_total_overlap)]
])
odds_ratio, p_value = stats.fisher_exact(contingency)

print('ChIP-seq x RNA-seq Integration Summary')
print('=' * 55)
print(f'DEGs (|log2FC| > 1, FDR < 0.05):           {n_degs:,}')
print(f'Differential binding sites (FDR < 0.05):   {n_diff_peaks:,}')
print(f'  -> Gained binding (treatment):             {1680:,}')
print(f'  -> Lost binding (treatment):               {1120:,}')
print(f'Overlap: diff peaks at DEG promoters:       {n_total_overlap:,}')
print(f'  -> Gained peaks + upregulated genes:       {n_overlap_gained:,}')
print(f'  -> Lost peaks + downregulated genes:       {n_overlap_lost:,}')
print(f'\nFisher\'s Exact Test:')
print(f'  Odds ratio: {odds_ratio:.2f}')
print(f'  p-value:    {p_value:.2e}')
sig_str = 'significant' if p_value < 0.05 else 'not significant'
print(f'  Enrichment is {sig_str} (alpha=0.05)')

# Simulate gene category sets
upregulated_genes = set([f'gene_{i}' for i in range(1500)])
downregulated_genes = set([f'gene_{i}' for i in range(1500, 3000)])
gained_peak_genes = set([
    f'gene_{i}' for i in np.random.choice(
        range(n_all_genes), n_overlap_gained * 3, replace=False
    )
])
lost_peak_genes = set([
    f'gene_{i}' for i in np.random.choice(
        range(n_all_genes), n_overlap_lost * 3, replace=False
    )
])

up_with_gained  = upregulated_genes & gained_peak_genes
down_with_lost  = downregulated_genes & lost_peak_genes

print(f'\nHigh-confidence regulatory targets:')
print(f'  Upregulated DEGs + gained TF binding:   {len(up_with_gained):,}')
print(f'  Downregulated DEGs + lost TF binding:   {len(down_with_lost):,}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

categories_bar = [
    'Upregulated\nonly', 'Gained peak\n+ Upregulated',
    'Downregulated\nonly', 'Lost peak\n+ Downregulated'
]
bar_counts = [
    len(upregulated_genes) - len(up_with_gained), len(up_with_gained),
    len(downregulated_genes) - len(down_with_lost), len(down_with_lost)
]
bar_colors = ['#aec6cf', '#2196F3', '#ffb3b3', '#F44336']
bars = axes[0].bar(categories_bar, bar_counts, color=bar_colors,
                   edgecolor='white', lw=1.5)
for bar, count in zip(bars, bar_counts):
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
        str(count), ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('DEG x Differential Peak Overlap\n(ChIP-seq + RNA-seq Integration)')
axes[0].set_ylim(0, max(bar_counts) * 1.15)

concordant_up   = len(up_with_gained)
concordant_down = len(down_with_lost)
discordant_up   = len(downregulated_genes & gained_peak_genes)
discordant_down = len(upregulated_genes & lost_peak_genes)

concordance_data = {
    'Gained peak +\nUpregulated\n(concordant)':   concordant_up,
    'Gained peak +\nDownregulated\n(discordant)': discordant_up,
    'Lost peak +\nDownregulated\n(concordant)':   concordant_down,
    'Lost peak +\nUpregulated\n(discordant)':     discordant_down,
}
colors_conc = ['#2196F3', '#9E9E9E', '#F44336', '#9E9E9E']
axes[1].bar(concordance_data.keys(), concordance_data.values(),
            color=colors_conc, edgecolor='white')
axes[1].set_ylabel('Gene Count')
axes[1].set_title('Concordance: Binding Change\nvs Expression Change')
axes[1].tick_params(axis='x', labelsize=8)
for bar, count in zip(axes[1].patches, concordance_data.values()):
    axes[1].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
        str(count), ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.show()

print('\nConcordance analysis:')
print(f'  Concordant (gained->up or lost->down): {concordant_up + concordant_down}')
print(f'  Discordant (gained->down or lost->up): {discordant_up + discordant_down}')
print('  Note: discordant cases may reflect indirect or delayed regulation')


## Summary

This notebook demonstrated the downstream analytical workflow applied to ChIP-seq peaks after the processing pipeline in Notebook 1. Differential binding analysis with DiffBind identifies loci where transcription factor occupancy or histone modification level changes between biological conditions, producing a ranked list of gained and lost binding sites. Peak annotation with ChIPseeker contextualises these sites within known genome annotations — promoters, UTRs, introns, and intergenic regions — revealing whether the factor primarily acts at promoters or at distal regulatory elements.

| Goal | Tool | Key Output | Key Parameter |
|---|---|---|---|
| Differential binding | DiffBind (R) | Gained/lost peaks BED | FDR < 0.05, \|FC\| > 1.5 |
| Consensus peak set | DiffBind dba.count | Read count matrix | minMembers = 2 |
| Normalization | DiffBind dba.normalize | Normalized counts | DBA_NORM_RLE |
| Peak annotation | ChIPseeker annotatePeak | Annotated peaks TSV | tssRegion = ±2 kb |
| Genomic feature dist. | ChIPseeker plotAnnoPie | Pie chart | — |
| TSS distance plot | ChIPseeker plotDistToTSS | Distance histogram | — |
| GO enrichment | clusterProfiler enrichGO | Enriched GO terms | ont = 'BP' |
| Redundancy removal | clusterProfiler simplify | Non-redundant terms | cutoff = 0.7 |
| Peak-DEG overlap | bedtools intersect | Regulatory targets BED | -wa -wb |
| Overlap significance | Fisher's exact test | Odds ratio, p-value | — |

**Next steps:** Motif enrichment analysis (HOMER `findMotifsGenome.pl` or MEME-ChIP) to identify the DNA sequence preferences of the immunoprecipitated factor at gained vs lost peaks. Integration with ATAC-seq data from the same conditions (Module 23) adds chromatin accessibility information, enabling identification of accessible TF binding sites versus nucleosome-occluded regions.


## Exercises

1. **DiffBind Sample Sheet:** A ChIP-seq experiment has 3 conditions (Control, DrugA, DrugB) with 2 replicates each. Write the column headers and two example rows for a DiffBind sample sheet. What `categories` argument would you pass to `dba.contrast()`?

2. **Annotation Interpretation:** Using the simulated annotation pie chart in Section 2, calculate the percentage of peaks that are promoter-associated (≤ 3 kb from TSS). How would you expect this percentage to change for an H3K27ac ChIP-seq experiment (enhancer mark) compared to a TF ChIP-seq? Explain why.

3. **GO Background Selection:** A researcher annotates all 45,000 peaks in a CTCF ChIP-seq to genes, then runs `enrichGO()` using all 20,000 human genes as background. Why might this produce misleading results? What background set would be more appropriate and why?

4. **Integration Concordance:** In Section 4, some peaks show discordant patterns (e.g., gained TF binding + downregulated gene expression). List two biological explanations for why a gained activating TF binding event could be accompanied by decreased target gene expression.

5. **Fisher's Exact Test:** Using the contingency table values from Section 4 (n_degs=3000, n_diff_peaks=2800, n_overlap=353, n_all_genes=20000), manually calculate the expected overlap under independence and compare to observed. Interpret the odds ratio.


---[< Previous: ChIP-seq Processing Pipeline](01_chipseq_pipeline.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: ONT Data Processing >](../25_Long_Read_Sequencing/01_ont_processing.ipynb)---